In [1]:
import os
import torch
from transformers import AutoModel, AutoTokenizer

local_model_path = "/active-data/datasets/models/gLM2_650M"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModel.from_pretrained(
    local_model_path,
    dtype=dtype,
    trust_remote_code=True,
    local_files_only=True
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    local_model_path,
    trust_remote_code=True,
    local_files_only=True
)

/home/zhongshitong/miniconda3/envs/glm2_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 232/232 [00:06<00:00, 36.16it/s]
[transformers] gLM2Model LOAD REPORT from: /active-data/datasets/models/gLM2_650M
Key                        | Status     |  | 
---------------------------+------------+--+-
lm_head.norm.weight        | UNEXPECTED |  | 
lm_head.proj_output.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
#test
sequence = "<+>MALTKVEKRNRIKRRVRGKISGTQASPRLSVYKSNK<+>aatttaaggaa<->MLGIDNIERVKPGGLELVDRLVAVNRVTKVTKGGRAFGFSAIVVVGNED"

# Tokenize the sequence.
encodings = tokenizer([sequence], return_tensors='pt')
# Extract embeddings.
with torch.no_grad():
    embeddings = model(encodings.input_ids.cuda(), output_hidden_states=True).last_hidden_state

print(embeddings)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


tensor([[[-62.0000, -40.7500,  45.0000,  ...,   4.0000,  35.7500,  -9.8750],
         [-87.0000, -13.1250,  50.5000,  ...,  14.1875,  17.6250,   2.8750],
         [-68.0000,   4.6250,  -3.0625,  ...,  29.7500,  33.0000, -13.2500],
         ...,
         [-37.0000, -34.0000,  18.0000,  ...,  49.0000,  13.5000,   9.0000],
         [ -7.1250, -23.2500,  12.5625,  ..., -12.7500, -14.3750,   8.5000],
         [-29.7500,   3.5625, -33.2500,  ..., -18.3750, -62.5000,  40.7500]]],
       device='cuda:0', dtype=torch.bfloat16)


In [3]:
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

def sequence_size(seq_id):
    acc_n, contig, start_site, end_site = seq_id.split('-')
    return int(end_site) - int(start_site)

def sequence_extract(acc_n, contig, start_site=None, end_site=None):
    gbff_path = f"/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff"
    for record in SeqIO.parse(gbff_path, "genbank"):
        if record.id != contig:
            continue
        if start_site and end_site:
            return str(record.seq)[int(start_site):int(end_site)]
        else:
            return str(record.seq)

base_folder = f'/active-data/analysis_results/chr_pla/genus'
genus_name = 'Escherichia'

replicon_data = pd.read_csv(f'{base_folder}/statistics_records/{genus_name}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
replicon_data.rename(columns={'accession': 'sequence'}, inplace=True)

chr_frag = pd.read_csv(f'{base_folder}/kmer_chr_frag_samples/{genus_name}/chromosome_fragment_data.tsv', sep='\t')
chr_frag['category-pident_90'] = 'chromosome fragment'
chr_frag['size'] = chr_frag['sequence'].apply(sequence_size)

all_data = pd.concat([replicon_data[['sequence', 'size', 'category-pident_90']], chr_frag[['sequence', 'size', 'category-pident_90']]], ignore_index=True)

In [4]:
from Bio import SeqIO

random_seed = 42
cut_len = 50000
target_dir = f'{base_folder}/embedding_vector/{genus_name}/glm2_mean'
vector_dir = f'{target_dir}/random_seed_{random_seed}'
os.makedirs(vector_dir, exist_ok=True)
categories = ['intermediate replicon', 'typical plasmid', 'chromosome fragment', 'typical chromosome']
sample_info = []
for cat in categories:
    try:
        sample = all_data[all_data["category-pident_90"]==cat].sample(n=200, random_state=random_seed).copy().sample(frac=1, random_state=random_seed+1).reset_index(drop=True)
    except:
        sample = all_data[all_data["category-pident_90"]==cat].copy().sample(frac=1, random_state=random_seed+1).reset_index(drop=True)

    with tqdm(total = len(sample), desc=f'{genus_name}:{cat}:{random_seed}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for idx, row in sample.iterrows():
            info = row['sequence'].split('-')
            if len(info) == 2:
                acc_n, contig = info
                dna_seq = sequence_extract(acc_n, contig)
                start_site, end_site = 0, len(dna_seq)
            elif len(info) == 4:
                acc_n, contig, start_site, end_site = info
                dna_seq = sequence_extract(acc_n, contig, start_site, end_site)
            if len(dna_seq) > cut_len:
                n = len(dna_seq) - cut_len
                random.seed(idx + len(dna_seq))
                cut_start = random.randrange(0, n)
                cut_end = cut_start + cut_len
            else:
                cut_start, cut_end = 0, len(dna_seq)
                
            sequence = dna_seq[cut_start:cut_end]
            encodings = tokenizer([sequence], return_tensors='pt').to(device)
            with torch.no_grad():
                outputs = model(encodings.input_ids, output_hidden_states=True)
                embeddings = outputs.last_hidden_state
            avg_1d = embeddings.mean(dim=1)
            avg_np = avg_1d.detach().cpu().to(torch.float32).numpy()
            
            os.chdir(vector_dir)
            np.save(f"{row['sequence']}.npy", avg_np)
            sample_info.append(pd.DataFrame([dict(row) | {'cut_start': cut_start, 'cut_end': cut_end, 'length': len(sequence), 'full_size': len(sequence)==len(dna_seq)}]))
            pbar.update(1)

sample_info = pd.concat(sample_info, ignore_index=True)
sample_info.to_csv(f'{vector_dir}_samples_info.tsv', sep='\t', index=False)

Escherichia:typical chromosome:42: 100%|████████████████████████████| 200/200 [21:04<00:00, 6.32s/B]
